# Nonlinear static analysis with structsolve

The callables of `structsolve.Analysis`, of
[structsolve](https://saullocastro.github.io/structsolve/), for the cylinder
of [Nonlinear static analysis](ex_nonlinear_static.ipynb), see the section
Solving with structsolve of [Analyses](../analyses.rst). They loop over the
elements with `update_fint`, `update_KCNL` and `update_KG`, assemble, and
work on the unknown degrees of freedom `bu`, the partition of the free
degrees of freedom. The code is that of the test
`tests/test_nonlinear_axial_compression.py`:

The callables return the matrices and vectors of the unknown degrees of
freedom, the partition of the free degrees of freedom, with the solution
vector $\boldsymbol{c} = \boldsymbol{u}_u$ and the prescribed degrees of
freedom zero. `calc_kC` with `NLgeom=True` gives $\boldsymbol{K}_{C0} +
\boldsymbol{K}_{CNL} + \boldsymbol{K}_{GNL}$ and `calc_kG` gives
$\boldsymbol{K}_G$, such that their sum is the tangent stiffness matrix
$\boldsymbol{K}_T$ and the residual of structsolve is
$\boldsymbol{F}_{int}(\boldsymbol{u}) - \lambda \boldsymbol{F}_{ext}$, see
[Analyses](../analyses.rst). The value arrays of the sparse matrices are
zeroed in each call, since the update functions add to them.

In [1]:
import time
import numpy as np
from numpy import isclose, pi
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import splu, spilu, LinearOperator
from scipy.sparse.linalg import cg, spsolve
from composites import laminated_plate
from bfsccylinder import (BFSCCylinder, update_KC0, update_KCNL, update_KG,
        update_fint, DOF, DOUBLE, INT, KC0_SPARSE_SIZE, KCNL_SPARSE_SIZE,
        KG_SPARSE_SIZE)
from bfsccylinder.quadrature import get_points_weights
from bfsccylinder.utils import assign_constant_ABD
from structsolve import Analysis

In [2]:
class Callables(object):
    """The callables of structsolve.Analysis for the ``elements`` of a model
    under the external forces ``fext``.

    They loop over the elements with the update functions and assemble, and
    work on the unknown degrees of freedom ``bu`` only, the partition of the
    free dofs of structsolve: the solution vector of the solvers is ``c =
    u[bu]``, and the force vectors and matrices are restricted to ``bu``.
    The prescribed degrees of freedom are zero.
    """
    def __init__(self, elements, points, weights, KC0, bu, fext):
        self.elements = elements
        self.points = points
        self.weights = weights
        self.N = KC0.shape[0]
        self.KC0 = KC0
        self.KC0uu = KC0[bu, :][:, bu]
        self.bu = bu
        self.fext = fext
        num_elements = len(elements)
        self.KCNLr = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
        self.KCNLc = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
        self.KCNLv = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=DOUBLE)
        self.KGr = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=INT)
        self.KGc = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=INT)
        self.KGv = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=DOUBLE)

    def expand(self, c):
        """The displacements of the model from those of the unknown dofs."""
        u = np.zeros(self.N, dtype=DOUBLE)
        u[self.bu] = c
        return u

    def calc_fext(self, inc=1., silent=False):
        return inc*self.fext[self.bu]

    def calc_fint(self, c, silent=False):
        fint = np.zeros(self.N, dtype=DOUBLE)
        u = self.expand(c)
        for shell in self.elements:
            update_fint(u, shell, self.points, self.weights, fint)
        return fint[self.bu]

    def calc_kC(self, c=None, NLgeom=False, silent=False):
        if not NLgeom:
            return self.KC0uu
        # the generated code accumulates with "+=" over the integration
        # points, so the value arrays must be zeroed on every call. The row
        # and column arrays are rewritten identically each time, and could
        # be filled once for a fixed mesh
        self.KCNLv *= 0
        u = self.expand(c)
        for shell in self.elements:
            update_KCNL(u, shell, self.points, self.weights, self.KCNLr,
                        self.KCNLc, self.KCNLv)
        KCNL = coo_matrix((self.KCNLv, (self.KCNLr, self.KCNLc)),
                          shape=(self.N, self.N)).tocsc()
        # KCNL also carries the geometric term of the nonlinear membrane
        # stress, so that KG stays homogeneous of degree one in u and keeps
        # its linear buckling meaning. KC0 + KCNL + KG is d(fint)/du
        return (self.KC0 + KCNL)[self.bu, :][:, self.bu]

    def calc_kG(self, c=None, NLgeom=False, silent=False):
        self.KGv *= 0
        u = self.expand(c)
        for shell in self.elements:
            update_KG(u, shell, self.points, self.weights, self.KGr,
                      self.KGc, self.KGv)
        KG = coo_matrix((self.KGv, (self.KGr, self.KGc)),
                        shape=(self.N, self.N)).tocsc()
        return KG[self.bu, :][:, self.bu]

The analysis, at the end of the test, applies the load in one increment,
`Analysis.initialInc = 1.`, with at most seven corrections,
`Analysis.maxNumIter = 7`, and checks that the increment was not cut:

In [3]:
def test_nonlinear_axial_compression_load_controlled():
    # geometry
    #Arbocz, J., and Starnes, J. H., 2002, “On a High-Fidelity Hierarchical Approach to Buckling Load Calculations,” New Approaches to Structural Mechanics, Shells and Biological Structures, pp. 271–292.
    L = 0.3556 # m
    R = 0.20318603 # m
    ny = 4*16

    load = 50000. # N

    # geometry of the filament-wound cylinder
    circ = 2*pi*R # m

    nx = int(1.5*ny*L/circ)
    if (nx % 2) == 0:
        nx += 1
    print('nx, ny', nx, ny)

    # material properties
    E11 = 127.629e9
    E22 = 11.3074e9
    G12 = 6.00257e9
    nu12 = 0.3002
    stack = [45, -45, 0, 90, 90, 0, -45, 45]
    plyt =  0.00101539/len(stack)
    laminaprop = (E11, E22, nu12, G12, G12, G12)
    prop = laminated_plate(stack=stack, plyt=plyt, laminaprop=laminaprop)

    nids = 1 + np.arange(nx*(ny+1))
    nids_mesh = nids.reshape(nx, ny+1)
    # closing the cylinder by reassigning last row of node-ids
    nids_mesh[:, -1] = nids_mesh[:, 0]
    nids = np.unique(nids_mesh)
    nid_pos = dict(zip(nids, np.arange(len(nids))))

    xlin = np.linspace(0, L, nx)
    ytmp = np.linspace(0, circ, ny+1)
    ylin = np.linspace(0, circ-(ytmp[-1] - ytmp[-2]), ny)
    xmesh, ymesh = np.meshgrid(xlin, ylin)
    xmesh = xmesh.T
    ymesh = ymesh.T

    # getting nodes
    ncoords = np.vstack((xmesh.flatten(), ymesh.flatten())).T
    x = ncoords[:, 0]
    y = ncoords[:, 1]

    #NOTE it didn't improve the performance
    # sorting nodes aiming more sparsity
    #asort = np.argsort(x**2 + y**2)
    #for i, nid in enumerate(nids):
        #nid_pos[nid] = asort[i]

    n1s = nids_mesh[:-1, :-1].flatten()
    n2s = nids_mesh[1:, :-1].flatten()
    n3s = nids_mesh[1:, 1:].flatten()
    n4s = nids_mesh[:-1, 1:].flatten()

    # number of integration points within element (along xi and eta)
    #TODO investigate different number of integration points
    nint = 4
    points, weights = get_points_weights(nint=nint)

    num_elements = len(n1s)
    print('num_elements', num_elements)

    elements = []
    N = DOF*nx*ny
    init_k_KC0 = 0
    init_k_KCNL = 0
    init_k_KG = 0
    for n1, n2, n3, n4 in zip(n1s, n2s, n3s, n4s):
        shell = BFSCCylinder(nint)
        shell.c1 = DOF*nid_pos[n1]
        shell.c2 = DOF*nid_pos[n2]
        shell.c3 = DOF*nid_pos[n3]
        shell.c4 = DOF*nid_pos[n4]
        shell.R = R
        shell.lex = L/(nx-1)
        shell.ley = circ/ny
        assign_constant_ABD(shell, prop)
        shell.init_k_KC0 = init_k_KC0
        shell.init_k_KCNL = init_k_KCNL
        shell.init_k_KG = init_k_KG
        init_k_KC0 += KC0_SPARSE_SIZE
        init_k_KCNL += KCNL_SPARSE_SIZE
        init_k_KG += KG_SPARSE_SIZE
        elements.append(shell)

    KC0r = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
    KC0c = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
    KC0v = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=DOUBLE)
    for shell in elements:
        update_KC0(shell, points, weights, KC0r, KC0c, KC0v)
    KC0 = coo_matrix((KC0v, (KC0r, KC0c)), shape=(N, N)).tocsc()

    print('KC0 ok')

    # applying SS3 boundary conditions
    bk = np.zeros(N, dtype=bool)

    checkSS = isclose(x, 0) | isclose(x, L)
    bk[3::DOF] = checkSS
    bk[6::DOF] = checkSS
    check = isclose(x, L/2.)
    bk[0::DOF] = check
    bu = ~bk # same as np.logical_not, defining unknown DOFs
    #NOTE every prescribed DOF is zero here, which the expansion u[bu] = c of
    #     Callables assumes. A non-zero prescribed displacement would need
    #     the load factor dependent load vector of structsolve, calc_fext
    #     declaring lbd

    # axially compressive load applied at x=0 and x=L
    checkTopEdge = isclose(x, L)
    checkBottomEdge = isclose(x, 0)
    fext = np.zeros(N)
    fext[0::DOF][checkBottomEdge] = +load/ny
    assert np.isclose(fext.sum(), load)
    fext[0::DOF][checkTopEdge] = -load/ny
    assert np.isclose(fext.sum(), 0)

    # solving using the full Newton-Raphson method of structsolve, in one
    # load step: the tangent stiffness matrix calc_kC + calc_kG is computed
    # again at every iteration, not held fixed as a modified Newton-Raphson
    # would. That is what buys quadratic convergence, now that KC0 + KCNL +
    # KG is the exact Jacobian of fint. The predictor is the linear solution
    # of KC0uu, and the convergence criterion is Crisfield's, with the
    # residual and the forces scaled with the diagonal of KC0uu, to 1e-9.
    # A consistent tangent reaches 2.0e-9 in two iterations here
    callables = Callables(elements, points, weights, KC0, bu, fext)
    an = Analysis(calc_fext=callables.calc_fext,
                  calc_fint=callables.calc_fint,
                  calc_kC=callables.calc_kC,
                  calc_kG=callables.calc_kG)
    an.initialInc = 1.
    # at most 7 corrections, otherwise structsolve cuts the load step
    an.maxNumIter = 7
    increments, cs = an.static(NLgeom=True)
    if increments != [1.]:
        raise RuntimeError('Not converged!')

In [4]:
test_nonlinear_axial_compression_load_controlled()

nx, ny 27 64
num_elements 1664


KC0 ok
Started Non-Linear Static Analysis
	___________________________________________
	                                           
	Newton-Raphson solver
	___________________________________________
	Initializing...


	Step 1, attempting load factor 1.00000
		Iteration: 1, max(|R|) = 5.808e+00, zeta = 2.461e-02


		Iteration: 2, max(|R|) = 1.195e-01, zeta = 2.219e-04


		Iteration: 3, max(|R|) = 3.197e-07, zeta = 2.021e-09


		Iteration: 4, max(|R|) = 4.178e-12, zeta = 3.484e-15
		Converged at load factor 1.00000
Finished Non-Linear Static Analysis
	    load factor 1.00000


The solution is `u = callables.expand(cs[-1])`. With the default
convergence criterion of Crisfield, $\zeta \le 10^{-9}$, the iterations
give $\zeta$ = 2.5e-2, 2.2e-4, 2.0e-9 and then the round-off, about 1e-15,
the quadratic convergence of a consistent tangent stiffness matrix.

## Follower pressure

The callables of a follower pressure, `FollowerCallables`, receive the load
factor `inc`, see the section Solving with structsolve of
[Analyses](../analyses.rst). A cylinder clamped at one end and free at the
other, with an unsymmetric load stiffness matrix, solved with these
callables for both kinematics, checking the equilibrium with the whole
follower pressure and the quadratic convergence of the Newton-Raphson
iterations. The code is that of the test
`tests/test_follower_pressure_structsolve.py`:

**Theory.** With the follower pressure $\lambda p_0$ the residual and its
Jacobian are, see [Follower pressure](../follower_pressure.rst),

$$
\boldsymbol{R}(\boldsymbol{u}) = \boldsymbol{F}_{int}(\boldsymbol{u}) -
\lambda \boldsymbol{F}_p(\boldsymbol{u}), \qquad
\boldsymbol{K}_T = \boldsymbol{K}_{C0} + \boldsymbol{K}_{CNL} +
\boldsymbol{K}_{GNL} + \boldsymbol{K}_G + \boldsymbol{K}_F
$$

`calc_fext` gives the dead part $\lambda \boldsymbol{F}_p(\boldsymbol{0})$,
`calc_fint` subtracts the part that depends on the displacements,
$\lambda (\boldsymbol{F}_p(\boldsymbol{u}) -
\boldsymbol{F}_p(\boldsymbol{0}))$, of `update_fext_follower` with
`reference=0`, and `calc_kC` adds the load stiffness matrix
$\boldsymbol{K}_F$ of $\lambda p_0$. The load stiffness matrix is symmetric
only if $p$ is uniform and, along the contour of the loaded region, $w = 0$
or the in-plane displacement normal to the contour vanishes
[Hibbitt1979], [SchweizerhofRamm1984]. Here both conditions fail: the free
end $x = L$ and the pressure $p = -2 (1 + \cos(3 y/R)/2)$ bar, which varies
around the circumference, make $\boldsymbol{K}_F$ unsymmetric.

**What the test verifies.** A cylinder of $R = 1$ m, $L = 1$ m and $h = 10$
mm, isotropic with the $E$ and $\nu$ of
`tests/test_follower_pressure_validation.py`, all ten degrees of freedom
constrained at the nodes of $x = 0$, solved in two increments of the load
factor, 0.5 and 1, for both kinematics. No published value is checked; the
test asserts that

- both increments converge without a cut;
- the solution is in equilibrium with the whole follower pressure,
  $\| (\boldsymbol{F}_{int} - \boldsymbol{F}_p)_u \| < 10^{-8} \|
  (\boldsymbol{F}_p)_u \|$, with $\boldsymbol{F}_p$ at the final
  displacements;
- the part of the load that follows the deformation is not negligible,
  $\| \boldsymbol{F}_p(\boldsymbol{u}) - \boldsymbol{F}_p(\boldsymbol{0}) \|
  > 10^{-3} \| \boldsymbol{F}_p(\boldsymbol{0}) \|$;
- $\boldsymbol{K}_F$ is unsymmetric, $\max |\boldsymbol{K}_F -
  \boldsymbol{K}_F^\top| > 10^{-3} \max |\boldsymbol{K}_F|$;
- the iterations converge quadratically: Crisfield's $\zeta$
  [Crisfield1981] ends below $10^{-9}$ in at most six iterations, the last
  one below $\max(10 \zeta_{prev}^2, 10^{-13})$. This happens only when the
  unsymmetric $\boldsymbol{K}_T$ is the Jacobian of the residual.

In [5]:
import sys

# the helper functions of the tests are imported from the folder tests/
sys.path.insert(0, '../tests')

In [6]:
import numpy as np
import pytest
from numpy import isclose
from scipy.sparse import coo_matrix
from composites import isotropic_plate
from structsolve import Analysis
from structsolve.convergence import crisfield_weights, crisfield_zeta
from test_follower_pressure_validation import (KINEMATICS, E, NU,
        cylinder_model, assemble)

In [7]:
class FollowerCallables(object):
    r"""The callables of structsolve.Analysis for the ``elements`` of the
    module ``mod`` under the follower pressures ``pressures``, one array of
    shape ``(nint, nint)`` per element, times the load factor

    They loop over the elements with the update functions and assemble, and
    work on the unknown degrees of freedom ``bu`` only, the partition of the
    free dofs of structsolve: the solution vector of the solvers is ``c =
    u[bu]``, and the force vectors and matrices are restricted to ``bu``.
    ``quadratic`` is the truncation of the area vector. ``calc_fint`` also
    keeps the load factor and the internal forces of each iteration, for the
    residual history.
    """
    def __init__(self, mod, elements, points, weights, N, bu, pressures,
                 quadratic=0):
        self.mod = mod
        self.elements = elements
        self.points = points
        self.weights = weights
        self.N = N
        self.bu = bu
        self.pressures = pressures
        self.quadratic = quadratic
        self.iterations = []
        self.KC0 = assemble(mod, elements, mod.update_KC0, 'KC0', N,
                            (points, weights))
        num_elements = len(elements)
        for i, shell in enumerate(elements):
            shell.init_k_KCNL = i*mod.KCNL_SPARSE_SIZE
            shell.init_k_KG = i*mod.KG_SPARSE_SIZE
            shell.init_k_KCfollower = i*mod.KCfollower_SPARSE_SIZE
        size = mod.KCNL_SPARSE_SIZE*num_elements
        self.KCNLr = np.zeros(size, dtype=mod.INT)
        self.KCNLc = np.zeros(size, dtype=mod.INT)
        self.KCNLv = np.zeros(size, dtype=mod.DOUBLE)
        size = mod.KG_SPARSE_SIZE*num_elements
        self.KGr = np.zeros(size, dtype=mod.INT)
        self.KGc = np.zeros(size, dtype=mod.INT)
        self.KGv = np.zeros(size, dtype=mod.DOUBLE)
        size = mod.KCfollower_SPARSE_SIZE*num_elements
        self.KCfr = np.zeros(size, dtype=mod.INT)
        self.KCfc = np.zeros(size, dtype=mod.INT)
        self.KCfv = np.zeros(size, dtype=mod.DOUBLE)
        # the dead part of the pressure, the force vector at u = 0
        self.fp0 = np.zeros(N, dtype=mod.DOUBLE)
        for shell, p in zip(elements, pressures):
            mod.update_fext_follower(np.zeros(N), shell, points, weights, p,
                                     self.fp0, quadratic)

    def expand(self, c):
        """The displacements of the model from those of the unknown dofs."""
        u = np.zeros(self.N, dtype=self.mod.DOUBLE)
        u[self.bu] = c
        return u

    def calc_fext(self, inc=1., silent=False):
        return inc*self.fp0[self.bu]

    def calc_fint(self, c, inc=1., silent=False):
        mod = self.mod
        u = self.expand(c)
        fint = np.zeros(self.N, dtype=mod.DOUBLE)
        for shell, p in zip(self.elements, self.pressures):
            mod.update_fint(u, shell, self.points, self.weights, fint)
            # minus the part of the follower pressure that depends on u
            mod.update_fext_follower(u, shell, self.points, self.weights,
                                     -inc*p, fint, self.quadratic, 0)
        self.iterations.append((inc, fint[self.bu]))
        return fint[self.bu]

    def calc_kC(self, c=None, NLgeom=False, inc=1., silent=False):
        bu = self.bu
        if not NLgeom:
            return self.KC0[bu, :][:, bu]
        mod = self.mod
        u = self.expand(c)
        self.KCNLv *= 0
        self.KCfv *= 0
        for shell, p in zip(self.elements, self.pressures):
            mod.update_KCNL(u, shell, self.points, self.weights, self.KCNLr,
                            self.KCNLc, self.KCNLv)
            # the load stiffness matrix of the pressure inc*p
            mod.update_KCfollower(u, shell, self.points, self.weights,
                                  inc*p, self.KCfr, self.KCfc, self.KCfv,
                                  self.quadratic)
        KCNL = coo_matrix((self.KCNLv, (self.KCNLr, self.KCNLc)),
                          shape=(self.N, self.N))
        KCf = coo_matrix((self.KCfv, (self.KCfr, self.KCfc)),
                         shape=(self.N, self.N))
        return (self.KC0 + KCNL + KCf).tocsc()[bu, :][:, bu]

    def calc_kG(self, c=None, NLgeom=False, silent=False):
        mod = self.mod
        u = self.expand(c)
        self.KGv *= 0
        for shell in self.elements:
            mod.update_KG(u, shell, self.points, self.weights, self.KGr,
                          self.KGc, self.KGv)
        KG = coo_matrix((self.KGv, (self.KGr, self.KGc)),
                        shape=(self.N, self.N))
        return KG.tocsc()[self.bu, :][:, self.bu]

In [8]:
def test_clamped_free_cylinder(kinematics):
    r"""Cylinder of radius 1 m, length 1 m and thickness 10 mm, clamped at
    `x = 0` and free at `x = L`, under the external follower pressure `p =
    -2 (1 + \cos(3 y/R)/2)` bar, in two equal load increments"""
    R, L, h = 1., 1., 0.01
    prop = isotropic_plate(thickness=h, E=E, nu=NU)
    mod, elements, points, weights, x, y, N = cylinder_model(kinematics, L,
            R, prop, nx=7, ny=24)
    DOF = mod.DOF
    bk = np.zeros(N, dtype=bool)
    for i in range(DOF):
        bk[i::DOF] |= isclose(x, 0)
    bu = ~bk
    nint = len(points)
    # p[i, j] at xi = points[i] and eta = points[j], from the circumferential
    # coordinate of node 1 of each element
    pressures = []
    for shell in elements:
        yj = y[shell.n1] + shell.ley*(points + 1)/2
        p = -2.e5*(1 + 0.5*np.cos(3*yj/R))
        pressures.append(np.ascontiguousarray(np.ones((nint, 1))*p[None, :]))

    callables = FollowerCallables(mod, elements, points, weights, N, bu,
                                  pressures)
    an = Analysis(calc_fext=callables.calc_fext,
                  calc_fint=callables.calc_fint,
                  calc_kC=callables.calc_kC,
                  calc_kG=callables.calc_kG)
    an.initialInc = 0.5
    an.maxInc = 0.5
    increments, cs = an.static(NLgeom=True)
    # every increment converged, none was cut
    assert np.allclose(increments, [0.5, 1.]), increments
    u = callables.expand(cs[-1])
    print('max |w|/h', np.abs(u[6::DOF]).max()/h)

    # equilibrium with the whole follower pressure, F_p(u) of the pressure p
    fint = np.zeros(N)
    fp = np.zeros(N)
    for shell, p in zip(elements, pressures):
        mod.update_fint(u, shell, points, weights, fint)
        mod.update_fext_follower(u, shell, points, weights, p, fp)
    residual = np.linalg.norm((fint - fp)[bu])/np.linalg.norm(fp[bu])
    print('relative residual', residual)
    assert residual < 1.e-8
    # the follower part of the load is not negligible
    assert np.linalg.norm((fp - callables.fp0)[bu]) > 1.e-3*np.linalg.norm(
            callables.fp0[bu])

    # the free end makes the load stiffness matrix unsymmetric
    KC = callables.calc_kC(cs[-1], NLgeom=True, inc=1.)
    KCnl = callables.calc_kC(cs[-1], NLgeom=True, inc=0.)
    KCf = (KC - KCnl).toarray()
    asym = np.abs(KCf - KCf.T).max()/np.abs(KCf).max()
    print('asymmetry of the load stiffness matrix', asym)
    assert asym > 1.e-3

    # quadratic convergence in each increment: Crisfield's zeta, the
    # criterion of structsolve, is squared by the last iteration, down to
    # the round-off
    w = crisfield_weights(callables.KC0[bu, :][:, bu])
    f = callables.fp0[bu]
    for lbd in increments:
        history = [crisfield_zeta(inc*f - fi, inc*f, fi, w)
                   for inc, fi in callables.iterations if inc == lbd]
        print('load factor', lbd, 'zeta', history)
        assert history[-1] < 1.e-9
        assert len(history) <= 6
        prev, curr = history[-2], history[-1]
        assert curr < max(10.*prev**2, 1.e-13), history

In [9]:
for kinematics in sorted(KINEMATICS):
    test_clamped_free_cylinder(kinematics)

Started Non-Linear Static Analysis
	___________________________________________
	                                           
	Newton-Raphson solver
	___________________________________________
	Initializing...
	Step 1, attempting load factor 0.50000
		Iteration: 1, max(|R|) = 9.725e+02, zeta = 2.968e-02


		Iteration: 2, max(|R|) = 3.290e+00, zeta = 1.612e-04


		Iteration: 3, max(|R|) = 2.833e-04, zeta = 1.715e-08


		Iteration: 4, max(|R|) = 8.983e-11, zeta = 1.535e-15
		Converged at load factor 0.50000
	Step 2, attempting load factor 1.00000
		Iteration: 1, max(|R|) = 1.989e+03, zeta = 3.117e-02


		Iteration: 2, max(|R|) = 4.744e+01, zeta = 1.353e-03


		Iteration: 3, max(|R|) = 7.309e-02, zeta = 2.064e-06


... (6 outputs left out)

		Iteration: 2, max(|R|) = 3.198e+01, zeta = 9.283e-04


		Iteration: 3, max(|R|) = 3.122e-02, zeta = 8.804e-07


		Iteration: 4, max(|R|) = 9.294e-08, zeta = 2.170e-12
		Converged at load factor 1.00000
Finished Non-Linear Static Analysis
	    load factor 1.00000
max |w|/h 0.39086687832744405
relative residual 4.7224813641061374e-12


asymmetry of the load stiffness matrix 2.0
load factor 0.5 zeta [0.02435330677001796, 0.00011273424226557484, 8.065847998248474e-09, 1.508913498641843e-15]
load factor 1.0 zeta [0.02535503330792601, 0.0009282563727527679, 8.804454100732186e-07, 2.1701603038416044e-12]


## References

- **[Crisfield1981]** Crisfield, M. A., A fast incremental/iterative solution procedure that
  handles "snap-through", Computers & Structures, 13(1-3), 55-62, 1981,
  doi:10.1016/0045-7949(81)90108-5.
- **[Hibbitt1979]** Hibbitt, H. D., Some follower forces and load stiffness, International
  Journal for Numerical Methods in Engineering, 14(6), 937-941, 1979,
  doi:10.1002/nme.1620140613.
- **[SchweizerhofRamm1984]** Schweizerhof, K., and Ramm, E., Displacement dependent pressure loads in
  nonlinear finite element analyses, Computers & Structures, 18(6),
  1099-1114, 1984, doi:10.1016/0045-7949(84)90154-8.